# P3 — Paths are messages

The bridge between P1 (over-squashing) and P2 (counting paths). After `g` layers, a message from `i` reaches
`j` **once per length-`g` path**. So `n_g(i,j)` is exactly *how many copies of `i`'s information land at `j`* —
all crammed into `j`'s one vector.

**Figures:** (1) message pile-up vs depth, (2) raw vs de-duplicated counts, (3) `A^g` vs `M_g` heatmaps.

In [ ]:
import torch, numpy as np
from oversquash import viz
from oversquash.data_bottleneck import make_bottleneck_graph
from oversquash.ideal_bridge import walk_operators
K, M = 5, 4

## Figure 1 — messages pile up (= the path count)

Same explosion as P1, now read as *message multiplicity*: `K·M^d` copies arrive at the target.

In [ ]:
depths = [1,2,3,4]; msgs = []
for d in depths:
    dd = make_bottleneck_graph(K, M, d, torch.Generator().manual_seed(0))
    raw, _ = walk_operators(dd.edge_index.numpy(), dd.num_nodes, max_length=d+1)
    t = int(dd.root_mask.nonzero()[0]); msgs.append(int(raw[d][:, t].sum()))
viz.plot_message_explosion(depths, msgs, 'messages arriving at the target', 'depth', 'messages (log)')

## Figure 2 — redundant vs distinct (raw vs kQ/I)

Many of those messages are **redundant** (interchangeable middle nodes carry the same content). The quotient
`kQ/I` merges equivalent paths, collapsing the count from `K·M^d` down to ~`K`.

In [ ]:
raw_d, eff_d = [], []
for d in depths:
    dd = make_bottleneck_graph(K, M, d, torch.Generator().manual_seed(0))
    raw, eff = walk_operators(dd.edge_index.numpy(), dd.num_nodes, max_length=d+1)
    t = int(dd.root_mask.nonzero()[0])
    raw_d.append(raw[d][:, t].sum()); eff_d.append(eff[d][:, t].sum())
viz.plot_raw_vs_eff(depths, raw_d, eff_d, 'raw vs de-duplicated messages', 'depth', 'messages (log)')

## Figure 3 — the two operators side by side

`A^g` (raw walk counts) vs `M_g = dim(e_i·(kQ/I)_g·e_j)` (de-duplicated). Same support, far fewer 'copies' on
the right.

**The plot twist (proved in P4–P5):** merging redundant messages is the *wrong* fix — it throws away signal.
The right fix is to keep all paths but **learn how much to weight each** (attention).

In [ ]:
dd = make_bottleneck_graph(K, M, 3, torch.Generator().manual_seed(0))
raw, eff = walk_operators(dd.edge_index.numpy(), dd.num_nodes, max_length=4)
viz.plot_two_heatmaps(raw[3], eff[3], titles=('A^4 (raw)', 'M_4 (kQ/I, de-duplicated)'))